# Lesson 03 - Structured support triage

**Level:** intermediate | **Demo time:** 8 minutes | **Builds on:** Lesson 01

Natural-language answers are ideal for people. Applications often need a predictable object for routing, reporting, or automation. MAF can ask compatible model clients for a response that conforms to a Pydantic model.

## Learning objectives

- Define a support-ticket schema with Pydantic.
- Request structured output at run time.
- Use validated fields for deterministic routing.

## Why not parse prose?

String splitting and regular expressions are brittle when wording changes. A response format gives the model an explicit JSON schema, and MAF returns the parsed object in `AgentResponse.value`.

Structured output improves shape validation; it does not prove that every extracted fact is true. Continue grounding factual decisions with tools and business rules.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class SupportTicket(BaseModel):
    """Normalized fields produced by the triage agent."""

    category: Literal["order_status", "return", "refund", "product_help", "other"]
    priority: Literal["low", "normal", "high"]
    order_id: str | None = Field(default=None, description="Order ID exactly as provided by the customer.")
    summary: str = Field(description="One-sentence issue summary.")
    requires_human: bool = Field(description="Whether policy, risk, or missing information needs a person.")

In [ ]:
from agent_framework import Agent

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
credential = create_credential()

triage_agent = Agent(
    client=create_chat_client(settings, credential),
    name="ticket_classifier",
    description="Converts customer messages into validated support tickets.",
    instructions=(
        "Classify the customer's request using only the supplied text. "
        "Use high priority for safety concerns, fraud, or a complete service outage. "
        "Set requires_human when a consequential decision or missing identity check is involved."
    ),
)

In [ ]:
customer_message = (
    "I was charged twice for ORD-1001. Please refund the duplicate charge today."
)

# customer_message = "Can you provide me with the status of order ORD-1001?"

response = await triage_agent.run(
    customer_message,
    options={"response_format": SupportTicket},
)

ticket = response.value
if ticket is None:
    raise RuntimeError("The model returned no structured ticket.")

ticket.model_dump()

The object is a real `SupportTicket`, not an unvalidated dictionary. Ordinary Python can now make a deterministic routing decision.

In [ ]:
queue_by_category = {
    "order_status": "fulfillment",
    "return": "returns",
    "refund": "billing",
    "product_help": "technical-support",
    "other": "general-support",
}

destination = "human-review" if ticket.requires_human else queue_by_category[ticket.category]
print(f"Route to: {destination}")
print(f"Priority: {ticket.priority}")
print(f"Summary: {ticket.summary}")

## Design trade-offs

- Put stable fields in the schema; avoid a giant model that changes every week.
- Use enums (`Literal`) when downstream code depends on a bounded set of values.
- Treat model classifications as recommendations when business risk is high.
- Version schemas and evaluation data together when deploying changes.

In [ ]:
credential.close()

## Recap

We can now ground inputs with tools and validate outputs with schemas. Lesson 04 adds a reusable `AgentSession` so follow-up questions retain context.